# 07 モデルの改善：設定の比較

`04_train` のモデル（v1）は、成功基準の 2 つ目（上限より下の行の MAE）に届かなかった。`06_error_analysis` で選んだ 2 つの案を、
**学習と検証の年（2018〜2024）だけ** で比べて、テストの年で評価する設定を 1 つ選ぶ。手順と決まりは `docs/plans/17-model-improvement.md`。

- 比べる設定は 6 つ（`ml/improve.py` の `SETTINGS`）。特徴量と基本の設定は v1 と同じで、1 か所ずつ変える。
  - `weighted_2`・`weighted_4`・`weighted_8`：正解が上限より下の行の重みを 2・4・8 にする。
  - `two_stage_50`・`two_stage_40`・`two_stage_30`：「上限より下か」の分類と、上限より下の行だけの回帰。分類の確率が閾値（0.5・0.4・0.3）以上なら回帰の値、未満なら上限。
  - `current`：v1 と同じ設定（比べる相手。選ぶ対象にしない）。
- 検証は 2 つの年で順に行う（`ml/split.py` の `ROLLING_FOLDS`）：学習 2018〜2022 → 検証 2023、学習 2018〜2023 → 検証 2024。
- 選ぶ決まり（`ml/improve.py` の `select`。結果を見る前に決めた）：
  1. 都市 × 検証の年の 4 つすべてで、基準 1（全体の MAE が `always_cap` より 10% 以上小さい）を満たす設定だけ残す。
  2. 点は、基準 2 の改善の割合（上限より下の行の MAE が `rule_of_thumb` より小さい割合）の、4 つのうち最も小さい値。
  3. 残った中で点が最も大きいものを選ぶ。
  4. その点が `current` の点より大きくなければ、選ばない（テストの年は評価しない）。
- 選んだ設定の、学習 2018〜2023・検証 2024 のモデルを、そのまま `models/cloud_base.joblib` に保存する（学習し直さない）。
- **テストの年（2025）は、読み込んだ直後に落とす**（ここでは見ない。`08_evaluate_improved` で 1 回だけ使う）。
- 既定は **サンプル**（`tests/fixtures/data_raw/` の 2 日ぶん。動くことの確認用で、数値に意味はない。一時的な場所に書く）。
  コミットしてある `reports/improvement.json` は、環境変数 `DATA_FETCH_MODE=full` を付けて全期間で実行したもの。外部とは通信しない。
- 時刻は UTC（分割の年も UTC）。単位は摂氏・メートル。

出典：Weather data by Open-Meteo.com（CC BY 4.0）、Iowa Environmental Mesonet（Iowa State University）。

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

import lightgbm as lgb
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from app.cities import City
from ml import (
    archive,
    baseline,
    dataset,
    evaluate,
    features,
    improve,
    metar,
    model,
    raw_store,
    split,
    target,
    train,
)

## 0. 設定・読み込み

In [ ]:
MODE = os.environ.get("DATA_FETCH_MODE", "sample")

if MODE == "full":
    OUTPUT_ROOT = ROOT
elif MODE == "sample":
    OUTPUT_ROOT = Path(tempfile.mkdtemp())
else:
    raise ValueError("DATA_FETCH_MODE must be 'sample' or 'full'")

# 選んだ設定のモデル（テストの年で評価する候補）。
MODEL_PATH = OUTPUT_ROOT / "models" / "cloud_base.joblib"
REPORT_PATH = OUTPUT_ROOT / "reports" / "improvement.json"
FOLDS = split.ROLLING_FOLDS
# 最終のモデルにする学習（spec の分割と同じ：学習 2018〜2023、検証 2024）。
FINAL_FOLD = list(FOLDS)[-1]
SETTINGS = {setting.name: setting for setting in improve.SETTINGS}

pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:.4f}".format)
print(f"mode={MODE}", list(FOLDS), list(SETTINGS))

In [ ]:
if MODE == "full":
    observations = dataset.load_dataset(ROOT / "data" / "processed" / "observations.parquet")
    dataset.validate_dataset(observations)
    # テストの年は、ここで落とす（設定を選ぶのに使わない）。
    is_test = split.rows_in_years(observations, split.SPLIT_YEARS[split.TEST])
    observations = observations[~is_test].reset_index(drop=True)
else:
    raw_root = ROOT / "tests" / "fixtures" / "data_raw"
    parts = {}
    for city in City:
        weather_path = raw_store.raw_path(raw_root, "open_meteo_archive", city, 2025, "json")
        metar_path = raw_store.raw_path(raw_root, "iem", city, 2025, "csv")
        parts[city.value] = (
            archive.parse_archive(weather_path.read_bytes().decode("utf-8")),
            metar.parse_metar(metar_path.read_bytes().decode("utf-8")),
        )
    observations = dataset.build_dataset(parts)
    dataset.validate_dataset(observations)

print(observations.shape, observations["time"].min(), observations["time"].max())

## 1. 特徴量・目的変数・比べる相手

`target_m` が欠損の行は、学習・評価から除く行。比べる相手は、成功基準に出てくる 2 つ（`always_cap`、`rule_of_thumb`）。

In [ ]:
X = features.features(observations)
y = target.target_m(observations)
baselines = {
    "rule_of_thumb": baseline.baseline_m(
        observations["temperature_c"], observations["dew_point_c"]
    ),
    "always_cap": pd.Series(target.CEILING_CAP_M, index=observations.index, dtype="float64"),
}

if MODE == "full":
    fold_rows = {
        name: (
            split.rows_in_years(observations, train_years),
            split.rows_in_years(observations, validation_years),
        )
        for name, (train_years, validation_years) in FOLDS.items()
    }
else:
    # fixture は 2025 年の 2 日ぶんだけなので、年で絞らず、同じ行で学習と評価をする。
    everything = pd.Series(True, index=observations.index)
    fold_rows = {name: (everything, everything) for name in FOLDS}

pd.DataFrame(
    {
        name: {
            "n_train": int(y[train_rows].notna().sum()),
            "n_validation": int(y[validation_rows].notna().sum()),
            "n_validation_below_cap": int((y[validation_rows] < target.CEILING_CAP_M).sum()),
        }
        for name, (train_rows, validation_rows) in fold_rows.items()
    }
).T

## 2. 学習と、検証の年での評価

検証の年ごとに、設定ごとのモデルを作り（`improve.fit_candidates`）、その検証の年で評価する。
成功基準と同じ判定（`evaluate.success_criteria`）を、都市ごとに出す。

In [ ]:
def criteria_for(rows, prediction):
    """`rows` の行の、都市ごとの成功基準の判定（丸める前の値）。"""
    result = {}
    for city in sorted(observations["city"][rows].unique()):
        city_rows = rows & (observations["city"] == city)
        result[city] = evaluate.success_criteria(
            model=evaluate.evaluate(y[city_rows], prediction[city_rows]),
            always_cap=evaluate.evaluate(y[city_rows], baselines["always_cap"][city_rows]),
            baseline=evaluate.evaluate(y[city_rows], baselines["rule_of_thumb"][city_rows]),
        )
    return result


def rounded(criteria):
    """`reports/` に書く形（割合は小数 4 桁）。"""
    return {
        city: {
            key: round(value, 4) if isinstance(value, float) else value
            for key, value in result.items()
        }
        for city, result in criteria.items()
    }


results = {name: {} for name in SETTINGS}
criteria = {name: {} for name in SETTINGS}
validation_baselines = {}
final_models = {}
for fold, (train_rows, validation_rows) in fold_rows.items():
    train_years, _ = FOLDS[fold]
    candidates = improve.fit_candidates(
        X[train_rows], y[train_rows], X[validation_rows], y[validation_rows], train_years
    )
    for name, candidate in candidates.items():
        predicted = model.predict_m(candidate, X)
        # 選ぶのに使う値（丸める前）。
        criteria[name][fold] = criteria_for(validation_rows, predicted)
        classifier = candidate.classifier
        results[name][fold] = {
            "n_train": int(y[train_rows].notna().sum()),
            "best_iteration": int(candidate.booster.best_iteration),
            "best_iteration_classifier": (
                None if classifier is None else int(classifier.best_iteration)
            ),
            "validation": evaluate.report(
                observations[validation_rows], y[validation_rows], predicted[validation_rows]
            ),
            "criteria": rounded(criteria[name][fold]),
        }
    validation_baselines[fold] = {
        label: evaluate.report(
            observations[validation_rows], y[validation_rows], prediction[validation_rows]
        )
        for label, prediction in baselines.items()
    }
    if fold == FINAL_FOLD:
        final_models = candidates

pd.DataFrame(
    [
        {
            "setting": name,
            "fold": fold,
            "city": city,
            "mae_m": result["validation"]["by_city"][city]["mae_m"],
            "mae_below_cap_m": result["validation"]["by_city"][city]["mae_below_cap_m"],
            **judged,
        }
        for name, by_fold in results.items()
        for fold, result in by_fold.items()
        for city, judged in result["criteria"].items()
    ]
)

## 3. 選ぶ

決まりは `improve.select`（上の 1〜4）。判定は丸める前の値で行う。

In [ ]:
chosen = improve.select(criteria)
selected = chosen["selected"]
selection = {
    name: {
        "eligible": item["eligible"],
        "score": None if item["score"] is None else round(item["score"], 4),
    }
    for name, item in chosen["selection"].items()
}

print("selected:", selected)
pd.DataFrame(selection).T

## 4. 保存

選んだ設定があれば、そのモデルを保存する（なければ、モデルは保存しない）。実行した時刻は入れない（再実行しても同じファイルになる）。

In [ ]:
if selected is not None:
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    model.save(final_models[selected], MODEL_PATH)
    # 読み直して、同じ予測になることを確かめる。
    loaded = model.load(MODEL_PATH)
    pd.testing.assert_series_equal(
        model.predict_m(loaded, X), model.predict_m(final_models[selected], X)
    )
    print(MODEL_PATH.name, MODEL_PATH.stat().st_size, loaded.model_version, loaded.train_years)

In [ ]:
report = {
    "mode": MODE,
    "cap_m": target.CEILING_CAP_M,
    "feature_names": list(features.FEATURE_NAMES),
    "lightgbm_version": lgb.__version__,
    "reference_model_version": model.MODEL_VERSION_V1,
    "params": train.PARAMS,
    "classifier_params": train.CLASSIFIER_PARAMS,
    "max_rounds": train.MAX_ROUNDS,
    "early_stopping_rounds": train.EARLY_STOPPING_ROUNDS,
    "folds": {
        name: {"train_years": list(train_years), "validation_years": list(validation_years)}
        for name, (train_years, validation_years) in FOLDS.items()
    },
    "settings": {
        name: {
            "kind": setting.kind,
            "below_cap_weight": setting.below_cap_weight,
            "below_cap_threshold": setting.below_cap_threshold,
        }
        for name, setting in SETTINGS.items()
    },
    "selection_rule": {
        "text": (
            "keep settings that meet criterion 1 in every city and fold; "
            "score is the smallest criterion 2 improvement; pick the highest score "
            "(the earlier setting on a tie) only if it is above the score of 'current'"
        ),
        "min_improvement": evaluate.SUCCESS_MIN_IMPROVEMENT,
    },
    "results": results,
    "validation_baselines": validation_baselines,
    "selection": selection,
    "selected": selected,
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
text = json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
REPORT_PATH.write_text(text, encoding="utf-8", newline="\n")
assert json.loads(REPORT_PATH.read_text(encoding="utf-8")) == report
print(REPORT_PATH.name, len(text))